# Generative AI with LSTM - Word-Level Text Generation
**Task:** Develop a text generator using Long Short-Term Memory (LSTM) recurrent neural networks, trained on Shakespeare's Complete Works using **word-level tokenization**.

## Pipeline Overview
1. **Dataset Loading & Preprocessing:** Automatic downloader, lowercasing, punctuation removal, word vocabulary construction (with `<unk>` token), and 90/10 train/validation sequence pair formulation.
2. **LSTM Model Architecture:** PyTorch Embedding layer (128-dim), multi-layer stacked LSTM (256-dim, 2 layers, 0.2 dropout), and dense Linear output layer.
3. **Training & Validation:** Training with cross-entropy loss, Adam optimizer, early stopping, perplexity logging, and checkpointing.
4. **Text Generation Engine:** Autoregressive word prediction using Temperature scaling, Top-K, and Top-P (nucleus) sampling.
5. **Bonus Experiments:** Comparative analysis of LSTM depth and sequence length on word prediction quality.

In [ ]:
# Step 0: Imports and Environment Setup
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

from data_preprocessing import prepare_data, download_dataset, load_and_preprocess_text, build_vocab
from model import LSTMTextGenerator
from train import run_training, plot_history
from generate import generate_text, load_model_checkpoint, generate_sample_suite

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Compute Device: {device}')

## Step 1: Dataset Loading & Preprocessing
We fetch Shakespeare's Complete Works, convert text to lower case, remove punctuation, tokenize at **word-level**, and construct input sequence pairs $(X, y)$ where $X$ is a sequence of 20 words and $y$ is the target next word.

In [ ]:
# Download & Inspect Preprocessed Text
data_path = download_dataset()
raw_text = load_and_preprocess_text(data_path, lowercase=True, remove_punctuation=True)
words = raw_text.split()
print(f'Dataset Total Words: {len(words):,}')
print('--- First 30 words sample ---')
print(' '.join(words[:30]))

# Build DataLoaders with 90/10 Split
train_loader, val_loader, metadata = prepare_data(data_path=data_path, seq_length=20, batch_size=64, val_split=0.1, level='word')
print(f'Word Vocabulary Size: {metadata["vocab_size"]:,}')
print(f'Train Batches: {len(train_loader)}, Val Batches: {len(val_loader)}')

## Step 2: Word-Level LSTM Model Design
The PyTorch model consists of:
- `nn.Embedding`: Maps word indices to dense vectors (dim=128)
- `nn.LSTM`: 2 stacked LSTM layers with hidden dimension 256 and dropout 0.2
- `nn.Linear`: Dense projection layer mapping hidden state to word vocabulary size (12,849 words)

In [ ]:
vocab_size = metadata['vocab_size']
model = LSTMTextGenerator(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_dim=256,
    num_layers=2,
    dropout=0.2
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'Total Trainable Parameters: {total_params:,}')

## Step 3: Model Training & Validation with Early Stopping
We train the model using **Adam Optimizer** and **Cross-Entropy Loss**. An **Early Stopping** monitor tracks validation loss and saves the best model checkpoint to `outputs/best_model.pt`.

In [ ]:
trained_model, metadata, history = run_training(
    epochs=5,
    seq_length=20,
    batch_size=64,
    learning_rate=0.002,
    embedding_dim=128,
    hidden_dim=256,
    num_layers=2,
    patience=3,
    save_path='outputs/best_model.pt',
    history_plot_path='outputs/training_history.png'
)

## Step 4: Word Text Generation & Sampling
Using the trained model, we generate new text by feeding word seed prompts (`"to be or"`, `"the king"`, `"my lord"`) and iteratively predicting the next word.

In [ ]:
seeds = ['to be or', 'the king', 'my lord', 'shall i compare']

for seed in seeds:
    print(f'\n========================================')
    print(f' Seed: "{seed}"')
    print(f'========================================')
    for temp in [0.2, 0.7, 1.2]:
        generated = generate_text(
            model=trained_model,
            metadata=metadata,
            seed_text=seed,
            gen_length=50,
            temperature=temp,
            top_k=40,
            top_p=0.9,
            device=device
        )
        print(f'\n[Temperature {temp}]:')
        print(generated)
        print('-' * 50)

## Step 5: Bonus - Word Architecture & Sequence Length Experiments
We evaluate how varying LSTM depth (1 vs 2 layers) and sequence length (10 vs 20 words) impacts model convergence and generation quality.

In [ ]:
from experiments import run_all_experiments

# Run comparative architectural benchmarks
run_all_experiments(save_dir='outputs')